In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

In [ ]:
import pandas as pd

from analysis.utils.stats import (
    KNOWN_METRIC_KEYWORDS,
    compute_clean_dirty_difference_statistics,
    compute_kl_target_correlations,
    compute_metric_statistics_by_group,
    get_benchmark_columns_by_keywords,
    summarize_kl_target_correlations,
)
from analysis.utils.vis import (
    plot_difference_statistics,
    plot_grouped_difference_statistics,
    plot_kl_correlation_barplot,
    plot_metrics_correlation,
    plot_run_comparisons,
)


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))

clean_results_path = "../clean_results/greedy/results.json"
clean_results = pd.read_json(clean_results_path, orient="records")

preview_results(clean_results)

In [ ]:
RELEVANT_FILES = [
    "anli",
    "blimp",
    "mastermind_easy",
    "metabench_arc",
    "metabench_hellaswag",
    "metabench_mmlu",
    "metabench_truthfulqa",
    "metabench_winogrande",
    "piqa",
    "toxigen",
    "wmdp",
]

In [ ]:
def filter_files(df: pd.DataFrame, relevant_files: list[str]) -> pd.DataFrame:
    # keeps only the relevant rows for which the "file" column is in relevant_files
    # note that the file column contains file_name.json, so we check if any of the relevant_files is a substring
    
    out = df.copy()
    out = out[out["file"].apply(lambda x: any(rel_file in x for rel_file in relevant_files))]
    return out


clean_results = filter_files(clean_results, RELEVANT_FILES)

In [ ]:
BENCHMARK_METRIC_SEP = " / "

def add_clean_model_name(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out["path"].astype(str).str.split("/")
    if (parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")
    out["model_name"] = parts.str[-2]
    return out

clean_results = add_clean_model_name(clean_results)
clean_results.head()

In [ ]:
clean_results.sample(10)

In [ ]:
print(clean_results.columns.tolist())

In [ ]:
# compute logprobs_normalized
# each entry in logprobs column contains list[float]
# each entry in num_tokens contains list[int]
# logprobs_normalized = logprob[i] / num_tokens[i]

clean_results["logprobs_normalized"] = clean_results.apply(
    lambda row: [lp / nt if nt > 0 else float('nan') for lp, nt in zip(row["logprobs"], row["choice_lengths"])], axis=1
)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import math
import numpy as np
import pandas as pd


@dataclass
class BenchmarkGTDistribution:
    benchmark: Any
    model_name: Any
    n_samples: int
    mean_score: float
    var_score: float
    std_score: float
    p_correct: np.ndarray          # shape: (m,)
    pmf_counts: np.ndarray         # shape: (m+1,), P[#correct = k]
    support_counts: np.ndarray     # [0, 1, ..., m]
    support_scores: np.ndarray     # support_counts / m
    pmf_scores: np.ndarray         # same as pmf_counts, but interpreted on support_scores
    cdf_scores: np.ndarray         # cumulative probability on support_scores


def _gold_prob_from_logprobs(logprobs: Sequence[float], gold_index: int) -> float:
    """
    Convert a list of log-probabilities/logits over answer options into the probability
    of the gold answer. This is robust both when the values are already log-probs
    and when they are unnormalized logits, because it normalizes via log-sum-exp.
    """
    arr = np.asarray(logprobs, dtype=np.float64)
    if arr.ndim != 1:
        raise ValueError(f"logprobs must be a 1D sequence, got shape {arr.shape}")
    if not (0 <= gold_index < len(arr)):
        raise IndexError(f"gold_index={gold_index} is out of bounds for length {len(arr)}")

    m = np.max(arr)
    log_denom = m + np.log(np.sum(np.exp(arr - m)))
    p = float(np.exp(arr[gold_index] - log_denom))
    return p


def _poisson_binomial_pmf(probs: Sequence[float]) -> np.ndarray:
    """
    Exact Poisson-binomial PMF via dynamic programming.
    Returns pmf[k] = P(sum Bernoullis = k), for k = 0..m.
    """
    probs = np.asarray(probs, dtype=np.float64)
    m = len(probs)

    pmf = np.zeros(m + 1, dtype=np.float64)
    pmf[0] = 1.0

    for p in probs:
        # Update backwards to avoid overwriting needed values.
        for k in range(m, 0, -1):
            pmf[k] = pmf[k] * (1.0 - p) + pmf[k - 1] * p
        pmf[0] *= (1.0 - p)

    # Numerical cleanup
    pmf = np.clip(pmf, 0.0, 1.0)
    pmf /= pmf.sum()
    return pmf


def characterize_benchmark_gt_distributions(
    df: pd.DataFrame,
    *,
    logprobs_col: str = "logprobs",
    benchmark_col: str = "benchmark",
    model_col: str = "model_name",
    gold_index_col: str = "gold_index",
    sample_index_col: str = "sample_index",
    require_unique_samples_per_group: bool = True,
) -> tuple[pd.DataFrame, Dict[Tuple[Any, Any], BenchmarkGTDistribution]]:
    """
    For each (benchmark, model_name) group, construct the benchmark-level ground-truth
    score distribution induced by per-sample answer probabilities.

    Assumptions:
    - Each row corresponds to one benchmark sample/question.
    - `logprobs_col` is a sequence of floats over answer options for that sample.
    - `gold_index_col` gives the correct answer index for that sample.
    - Sample correctness for question j is modeled as Bernoulli(p_j), where p_j is the
      model probability of the gold answer.
    - Final benchmark score is the average correctness over all samples in the group.

    Returns:
    1) summary_df: one row per (benchmark, model_name), with mean/var/std and count.
    2) distributions: dict keyed by (benchmark, model_name), whose values contain the
       exact Poisson-binomial distribution over final scores.

    Notes:
    - The exact final-score distribution is discrete:
          score in {0/m, 1/m, ..., m/m}
      with probabilities given by the Poisson-binomial PMF.
    - This is exact under independent per-sample draws from the model's choice
      distribution for each benchmark sample.
    """
    required = {
        logprobs_col,
        benchmark_col,
        model_col,
        gold_index_col,
        sample_index_col,
    }
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    work_df = df.copy()

    # Validate and compute p(correct) for each row.
    work_df["_p_correct"] = [
        _gold_prob_from_logprobs(lp, gi)
        for lp, gi in zip(work_df[logprobs_col], work_df[gold_index_col])
    ]

    summary_rows: List[Dict[str, Any]] = []
    distributions: Dict[Tuple[Any, Any], BenchmarkGTDistribution] = {}

    group_cols = [benchmark_col, model_col]

    for (benchmark, model_name), g in work_df.groupby(group_cols, sort=False):
        if require_unique_samples_per_group:
            dup_mask = g.duplicated(subset=[sample_index_col], keep=False)
            if dup_mask.any():
                dup_vals = g.loc[dup_mask, sample_index_col].tolist()
                raise ValueError(
                    f"Duplicate {sample_index_col} values found in group "
                    f"(benchmark={benchmark!r}, model_name={model_name!r}): {dup_vals}"
                )

        # Optional but helpful: sort by sample index for reproducibility.
        g = g.sort_values(sample_index_col)

        probs = g["_p_correct"].to_numpy(dtype=np.float64)
        m = len(probs)
        if m == 0:
            raise ValueError(
                f"Empty group encountered for (benchmark={benchmark!r}, model_name={model_name!r})."
            )

        mean_score = float(np.mean(probs))
        var_score = float(np.sum(probs * (1.0 - probs)) / (m * m))
        std_score = float(np.sqrt(var_score))

        pmf_counts = _poisson_binomial_pmf(probs)
        support_counts = np.arange(m + 1, dtype=np.int64)
        support_scores = support_counts / m
        pmf_scores = pmf_counts.copy()
        cdf_scores = np.cumsum(pmf_scores)

        dist = BenchmarkGTDistribution(
            benchmark=benchmark,
            model_name=model_name,
            n_samples=m,
            mean_score=mean_score,
            var_score=var_score,
            std_score=std_score,
            p_correct=probs,
            pmf_counts=pmf_counts,
            support_counts=support_counts,
            support_scores=support_scores,
            pmf_scores=pmf_scores,
            cdf_scores=cdf_scores,
        )

        distributions[(benchmark, model_name)] = dist
        summary_rows.append(
            {
                benchmark_col: benchmark,
                model_col: model_name,
                "n_samples": m,
                "mean_score": mean_score,
                "var_score": var_score,
                "std_score": std_score,
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    return summary_df, distributions


def upper_tail_probability(dist: BenchmarkGTDistribution, s: float) -> float:
    """
    Exact upper-tail probability P(score >= s) under the benchmark GT distribution.
    """
    mask = dist.support_scores >= s
    return float(dist.pmf_scores[mask].sum())


def lower_tail_probability(dist: BenchmarkGTDistribution, s: float) -> float:
    """
    Exact lower-tail probability P(score <= s) under the benchmark GT distribution.
    """
    mask = dist.support_scores <= s
    return float(dist.pmf_scores[mask].sum())


def two_sided_tail_probability(dist: BenchmarkGTDistribution, s: float) -> float:
    """
    Two-sided 'extremeness' score. Computes deviation d of s from the mean, and then
    returns P(score >= mean + d) + P(score <= mean - d) clipped to [0, 1]. 
    """
    mean = dist.mean_score
    d = abs(s - mean)
    upper_tail = upper_tail_probability(dist, mean + d)
    lower_tail = lower_tail_probability(dist, mean - d)
    p = upper_tail + lower_tail
    return float(min(1.0, p))

In [ ]:
summary_df, dists = characterize_benchmark_gt_distributions(
    clean_results,
    logprobs_col="logprobs",
)

print(summary_df)

In [ ]:
# sort by variance

summary_df = summary_df.sort_values("var_score", ascending=False).reset_index(drop=True)
summary_df

In [ ]:
model_name, bench_name = "Llama-2-7b-chat-hf", "anli_r1"

In [ ]:
dists.keys()

In [ ]:
summary_df[(summary_df["benchmark"] == bench_name) & (summary_df["model_name"] == model_name)]

In [ ]:
dist = dists[(bench_name, model_name)]


mean = summary_df[(summary_df["benchmark"] == bench_name) & (summary_df["model_name"] == model_name)]["mean_score"].values[0]

s = mean + 0.6 / 100
p_upper = upper_tail_probability(dist, s)
p_lower = lower_tail_probability(dist, s)
p_two_sided = two_sided_tail_probability(dist, s)

print(p_upper, p_lower, p_two_sided)